In [1]:
import pandas as pd
import numpy as np
import mwparserfromhell
import os
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.http import models
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
import shutil
from langchain_core.documents import Document
from qdrant_client.http.models import PointStruct
from langchain.vectorstores import Qdrant as LCQdrant


/mnt/shared/miniconda3/envs/ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_PATH = "data"
DOCS_PATH = os.path.join(DATA_PATH, "docs")
# EMBEDDINGS_PATH = "embeddings"
EMBEDDINGS_PATH = "embeddings_hybrid"


# Vector db

In [3]:
import torch
torch.cuda.is_available()

True

In [4]:
client = QdrantClient(path="qdrant")

In [5]:
sample_embedding = np.load(os.path.join(EMBEDDINGS_PATH, "3Dconnexion_input_devices_0.npy"))
sample_embedding.shape[0]

1024

In [6]:
collection_name = "freecad_docs_hybrid"

if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=sample_embedding.shape[0],
            distance=models.Distance.COSINE
        ),
    )


In [7]:
chunks_df = pd.read_csv(os.path.join(DATA_PATH, "chunks_df_hybrid.csv"))

In [8]:
def get_content(source, chunk_idx, df) -> str:
    cond = (df['source'] == source) & (df['chunk_id'] == chunk_idx)
    return df[cond]['content'].values[0]


def load_embeddings_with_payload():
    embeddings = []
    payload = []

    for filename in os.listdir(EMBEDDINGS_PATH):
        if filename.endswith(".npy"):
            embedding = np.load(os.path.join(EMBEDDINGS_PATH, filename))
            embeddings.append(embedding)

            filename = os.path.splitext(filename)[0]
            split = filename.split("_")
            
            source = "_".join(split[:-1]) + ".wikitext"
            chunk_idx = int(split[-1])            

            payload.append({
                "source": source,
                "chunk_idx": chunk_idx,
                "content": get_content(source, chunk_idx, chunks_df)
            })

    
    return embeddings, payload


def load_embeddings_with_doc():
    embeddings = []
    documents: list[dict] = []

    for filename in os.listdir(EMBEDDINGS_PATH):
        if filename.endswith(".npy"):
            embedding = np.load(os.path.join(EMBEDDINGS_PATH, filename))
            embeddings.append(embedding)

            filename = os.path.splitext(filename)[0]
            split = filename.split("_")
            
            source = "_".join(split[:-1]) + ".wikitext"
            chunk_idx = int(split[-1])

            documents.append({
                "page_content": get_content(source, chunk_idx, chunks_df),
                "metadata": {"source": source, "chunk_idx": chunk_idx}
            })

            # documents.append(Document(
            #     page_content=get_content(source, chunk_idx, chunks_df),
            #     metadata={"source": source, "chunk_idx": chunk_idx}
            # ))
    
    return embeddings, documents


In [9]:
embeddings, payload = load_embeddings_with_payload()

client.upload_collection(
    collection_name=collection_name,
    vectors=embeddings,
    payload=payload,
    ids=None,
    batch_size=32,
)

# Index for LC

In [4]:
client = QdrantClient(path="qdrant")

In [5]:
sample_embedding = np.load(os.path.join(EMBEDDINGS_PATH, "3Dconnexion_input_devices_0.npy"))
sample_embedding.shape[0]

1024

In [6]:
collection_name = "freecad_docs_lc"
if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=sample_embedding.shape[0],
            distance=models.Distance.COSINE
        ),
    )


In [7]:
chunks_df = pd.read_csv(os.path.join(DATA_PATH, "chunks_df.csv"))

In [8]:
embeddings, docs = load_embeddings_with_doc()

client.upload_collection(
    collection_name=collection_name,
    vectors=embeddings,
    payload=docs,
    ids=None,
    batch_size=32,
)

# Inference

In [7]:
embedding_model = SentenceTransformer("BAAI/bge-m3")

OutOfMemoryError: CUDA out of memory. Tried to allocate 978.00 MiB. GPU 0 has a total capacity of 7.62 GiB of which 922.50 MiB is free. Including non-PyTorch memory, this process has 6.71 GiB memory in use. Of the allocated memory 5.75 GiB is allocated by PyTorch, and 893.20 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [6]:
from langchain_huggingface.llms import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

qwen_model = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(
    qwen_model,
    trust_remote_code=True
)
model = AutoModelForCausalLM.from_pretrained(
    qwen_model,
    trust_remote_code=True,
    device_map=device
)

text_gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024
)
llm = HuggingFacePipeline(pipeline=text_gen)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
Device set to use cuda
